# Основы классификации: MLP и MNIST

Цель: построить MLP для классификации рукописных цифр MNIST
и сравнить активации ReLU и Sigmoid.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np


from pathlib import Path

def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").exists() and (candidate / "requirements.txt").exists():
            return candidate
    return current.parent if current.name == "notebooks" else current

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
FIGURES_DIR = REPO_ROOT / "figures" / "mlp_mnist"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / 'mnist').mkdir(parents=True, exist_ok=True)

 
print("Папка для графиков:", FIGURES_DIR)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Device: {device}')

## 1. Загрузка данных MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean/std
])

# Загрузка
full_train = torchvision.datasets.MNIST(root=str(DATA_DIR / 'mnist'), train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root=str(DATA_DIR / 'mnist'), train=False, download=True, transform=transform)

# Train / Validation split (50000 / 10000)
train_size = 50000
val_size = len(full_train) - train_size
train_dataset, val_dataset = random_split(
    full_train, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')

## 2. Визуализация примеров

In [ ]:
examples = iter(train_loader)
images, labels = next(examples)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(str(labels[i].item()))
    ax.axis('off')
plt.suptitle('Примеры из MNIST (28×28 → flatten 784)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mnist_examples.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Определение модели MLP

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_size=256, activation='relu'):
        super().__init__()
        act = nn.ReLU() if activation == 'relu' else nn.Sigmoid()
        self.net = nn.Sequential(
            nn.Flatten(),           # 28*28=784
            nn.Linear(784, hidden_size),
            act,
            nn.Linear(hidden_size, hidden_size // 2),
            act,
            nn.Linear(hidden_size // 2, 10)
        )

    def forward(self, x):
        return self.net(x)

# Быстрая проверка размерностей
model_check = MLP()
dummy = torch.zeros(4, 1, 28, 28)
print('Output shape:', model_check(dummy).shape)  # [4, 10]

## 4. Функции обучения и оценки

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = criterion(logits, y)
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total


def run_experiment(activation='relu', epochs=15, lr=1e-3):
    model = MLP(hidden_size=256, activation=activation).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc = eval_epoch(model, val_loader, criterion)
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        if epoch % 5 == 0 or epoch == 1:
            print(f'[{activation.upper():7s}] Epoch {epoch:2d}/{epochs} | '
                  f'Train loss: {tr_loss:.4f}, acc: {tr_acc:.4f} | '
                  f'Val loss: {vl_loss:.4f}, acc: {vl_acc:.4f}')

    test_loss, test_acc = eval_epoch(model, test_loader, criterion)
    print(f'\n>>> Test Accuracy ({activation}): {test_acc:.4f}\n')
    return history, model, test_acc

## 5. Обучение: ReLU vs Sigmoid

In [ ]:
EPOCHS = 15

history_relu, model_relu, test_acc_relu = run_experiment('relu', epochs=EPOCHS)
history_sigmoid, model_sigmoid, test_acc_sigmoid = run_experiment('sigmoid', epochs=EPOCHS)

## 6. Графики Loss и Accuracy

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- ReLU ---
axes[0, 0].plot(epochs_range, history_relu['train_loss'], label='Train')
axes[0, 0].plot(epochs_range, history_relu['val_loss'],   label='Validation')
axes[0, 0].set_title('ReLU — Loss')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()

axes[0, 1].plot(epochs_range, history_relu['train_acc'], label='Train')
axes[0, 1].plot(epochs_range, history_relu['val_acc'],   label='Validation')
axes[0, 1].set_title('ReLU — Accuracy')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()

# --- Sigmoid ---
axes[1, 0].plot(epochs_range, history_sigmoid['train_loss'], label='Train')
axes[1, 0].plot(epochs_range, history_sigmoid['val_loss'],   label='Validation')
axes[1, 0].set_title('Sigmoid — Loss')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()

axes[1, 1].plot(epochs_range, history_sigmoid['train_acc'], label='Train')
axes[1, 1].plot(epochs_range, history_sigmoid['val_acc'],   label='Validation')
axes[1, 1].set_title('Sigmoid — Accuracy')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()

plt.suptitle('Train vs Validation: ReLU vs Sigmoid', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Сравнение сходимости на одном графике

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Val Loss
ax1.plot(epochs_range, history_relu['val_loss'],    label='ReLU',    color='steelblue')
ax1.plot(epochs_range, history_sigmoid['val_loss'], label='Sigmoid', color='tomato')
ax1.set_title('Validation Loss: ReLU vs Sigmoid')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend()

# Val Accuracy
ax2.plot(epochs_range, history_relu['val_acc'],    label='ReLU',    color='steelblue')
ax2.plot(epochs_range, history_sigmoid['val_acc'], label='Sigmoid', color='tomato')
ax2.set_title('Validation Accuracy: ReLU vs Sigmoid')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'activation_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Final Test Accuracy — ReLU:    {test_acc_relu:.4f} ({test_acc_relu*100:.2f}%)')
print(f'Final Test Accuracy — Sigmoid: {test_acc_sigmoid:.4f} ({test_acc_sigmoid*100:.2f}%)')

## 8. Примеры предсказаний

In [ ]:
model_relu.eval()
test_images, test_labels = next(iter(test_loader))
with torch.no_grad():
    preds = model_relu(test_images.to(device)).argmax(1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(test_images[i].squeeze(), cmap='gray')
    color = 'green' if preds[i] == test_labels[i] else 'red'
    ax.set_title(f'P:{preds[i]} T:{test_labels[i]}', color=color, fontsize=8)
    ax.axis('off')
plt.suptitle('Предсказания MLP (ReLU) — зелёный=верно, красный=ошибка')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'test_predictions.png', dpi=120, bbox_inches='tight')
plt.show()

## Итоги

| Активация | Test Accuracy |
|-----------|---------------|
| ReLU      | см. вывод выше |
| Sigmoid   | см. вывод выше |

**Выводы:**
- ReLU обычно сходится быстрее из-за отсутствия проблемы затухающего градиента.
- Sigmoid насыщается при больших значениях, что замедляет обучение.
- Оба варианта превышают требуемый порог 70% на тесте.